In [4]:
import pandas as pd

# Load the training spreadsheet into a pandas DataFrame.
# 将训练表格读入 pandas 的 DataFrame，后续用 df 来查看和处理数据。
df = pd.read_excel("TrainDataset2026.xls")

# Check the number of rows and columns, then list the column names.
# 检查行列数，再查看列名，方便区分病人 ID、预测目标和输入特征。
print(df.shape)
print(df.columns.tolist())

(400, 121)
['ID', 'pCR (outcome)', 'RelapseFreeSurvival (outcome)', 'Age', 'ER', 'PgR', 'HER2', 'TrippleNegative', 'ChemoGrade', 'Proliferation', 'HistologyType', 'LNStatus', 'TumourStage', 'Gene', 'original_shape_Elongation', 'original_shape_Flatness', 'original_shape_LeastAxisLength', 'original_shape_MajorAxisLength', 'original_shape_Maximum2DDiameterColumn', 'original_shape_Maximum2DDiameterRow', 'original_shape_Maximum2DDiameterSlice', 'original_shape_Maximum3DDiameter', 'original_shape_MeshVolume', 'original_shape_MinorAxisLength', 'original_shape_Sphericity', 'original_shape_SurfaceArea', 'original_shape_SurfaceVolumeRatio', 'original_shape_VoxelVolume', 'original_firstorder_10Percentile', 'original_firstorder_90Percentile', 'original_firstorder_Energy', 'original_firstorder_Entropy', 'original_firstorder_InterquartileRange', 'original_firstorder_Kurtosis', 'original_firstorder_Maximum', 'original_firstorder_MeanAbsoluteDeviation', 'original_firstorder_Mean', 'original_firstorder

In [2]:
%pip install "xlrd>=2.0.1"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Count patients in each pCR class. Include blank labels in the count.
# 统计每种 pCR 标签有多少名患者。dropna=False 会把空白标签也显示出来。
# The spreadsheet uses 999 for missing values, so check whether 999 appears too.
# 表格用 999 表示缺失值，所以也要检查结果里有没有 999。
print(df["pCR (outcome)"].value_counts(dropna=False))

pCR (outcome)
0      311
1       84
999      5
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

# Keep only patients with a known pCR label (0 or 1).
# 只保留 pCR 标签为 0 或 1 的患者，原始 df 留给后续的 RFS 分析。
pcr_data = df.loc[df["pCR (outcome)"].isin([0, 1])].copy()

# Use pCR as the target. Exclude the patient ID and both outcomes from the inputs.
# 用 pCR 作为预测目标，输入特征中去掉病人 ID 和两个结局列。
y = pcr_data["pCR (outcome)"]
X = pcr_data.drop(columns=["ID", "pCR (outcome)", "RelapseFreeSurvival (outcome)"])

# Preserve the class proportions and fix the seed so teammates can reproduce the split.
# 保持两边的类别比例接近，并固定随机种子，方便组员复现同一划分。
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Check the class counts in each split.
# 检查训练集和验证集各有多少条 0 和 1。
print("Training labels:\n", y_train.value_counts().sort_index())
print("Validation labels:\n", y_val.value_counts().sort_index())

Training labels:
 pCR (outcome)
0    249
1     67
Name: count, dtype: int64
Validation labels:
 pCR (outcome)
0    62
1    17
Name: count, dtype: int64


In [7]:
%pip install scikit-learn

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ------------------------ --------------- 5.5/8.9 MB 30.5 MB/s eta 0:00:01
   ---------------------------------------- 8.9/8.9 MB 29.0 MB/s  0:00:00
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   ---- ----------------------------------- 4.5/41.3 MB 24.4 MB/s eta 0:00:02
   ------- -------------------------------- 7.6/41.3 MB 18.8 MB/s eta 0:00:02
   --------- ------------------------------ 9.7/41.3 MB 15.9 MB/s eta 0:00:02
   ----------- ---------------------------- 12.1/41.3 MB 14.5 MB/s eta 0:00:03
   ------------ --------------------------- 13.4/41.3 MB 13.1 MB/s eta 0:00:03
   ------------- -------------------------- 14.4/41.3 MB 11.5 MB/s eta 0:00:03



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# Convert the spreadsheet's missing-value marker to NaN in both splits.
# 把训练集和验证集特征里的缺失标记 999 转成 NaN，保留原来的变量。
X_train_clean = X_train.replace([999, "999"], float("nan"))
X_val_clean = X_val.replace([999, "999"], float("nan"))

# Count missing values in the training features and show the ten most affected columns.
# 统计训练集各特征的缺失数量，显示缺失最多的十列。
missing_counts = X_train_clean.isna().sum().sort_values(ascending=False)
print(missing_counts[missing_counts > 0].head(10))

# Check the three required features even if they are outside the top ten.
# 单独检查三个必留特征，即使它们没出现在前十列里。
print("\nRequired features / 必留特征")
print(missing_counts[["ER", "HER2", "Gene"]])

Gene             70
ChemoGrade        1
HistologyType     1
LNStatus          1
dtype: int64

Required features / 必留特征
ER       0
HER2     0
Gene    70
dtype: int64


In [10]:
# Inspect the values in the training columns that contain missing data.
# 查看训练集中有缺失的列，了解每列的取值和分布。
for column in ["Gene", "ChemoGrade", "HistologyType", "LNStatus"]:
    values = X_train_clean[column]

    # Show the data type, number of distinct known values, and most common values.
    # 显示数据类型、非缺失的不同取值数，以及最常见的取值。
    print(f"\n{column} | dtype: {values.dtype} | unique: {values.nunique()}")
    print(values.value_counts(dropna=False).head(10))


Gene | dtype: float64 | unique: 2
Gene
0.0    152
1.0     94
NaN     70
Name: count, dtype: int64

ChemoGrade | dtype: float64 | unique: 3
ChemoGrade
2.0    187
3.0    127
NaN      1
1.0      1
Name: count, dtype: int64

HistologyType | dtype: float64 | unique: 2
HistologyType
1.0    268
2.0     47
NaN      1
Name: count, dtype: int64

LNStatus | dtype: float64 | unique: 2
LNStatus
1.0    166
0.0    149
NaN      1
Name: count, dtype: int64


In [11]:
# Copy the feature tables so we can reuse the versions with missing values.
# 复制两份特征表，方便以后对比别的填补方法。
X_train_filled = X_train_clean.copy()
X_val_filled = X_val_clean.copy()

# Use -1 as a separate category for missing Gene values.
# 用 -1 标记 Gene 缺失，后续编码时把它当成一个独立类别。
X_train_filled["Gene"] = X_train_filled["Gene"].fillna(-1)
X_val_filled["Gene"] = X_val_filled["Gene"].fillna(-1)

# Learn the most frequent value of each column from the training set.
# 每列的众数只从训练集计算，再用同一个值填补训练集和验证集。
for column in ["ChemoGrade", "HistologyType", "LNStatus"]:
    most_frequent = X_train_clean[column].mode().iloc[0]
    X_train_filled[column] = X_train_filled[column].fillna(most_frequent)
    X_val_filled[column] = X_val_filled[column].fillna(most_frequent)

# Check the Gene categories and count any remaining missing values.
# 检查 Gene 的类别分布，以及两份特征表中还剩多少缺失值。
print(X_train_filled["Gene"].value_counts(dropna=False).sort_index())
print("Remaining missing values (train):", X_train_filled.isna().sum().sum())
print("Remaining missing values (validation):", X_val_filled.isna().sum().sum())

Gene
-1.0     70
 0.0    152
 1.0     94
Name: count, dtype: int64
Remaining missing values (train): 0
Remaining missing values (validation): 5


In [12]:
# Find the validation columns that still contain missing values.
# 找出验证集中仍有缺失值的列。
remaining = X_val_filled.isna().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)
print(remaining)

# Inspect the corresponding training values to help choose an imputation method.
# 查看这些列在训练集中的取值，帮助决定怎么填补。
for column in remaining.index:
    values = X_train_filled[column]
    print(f"\n{column} | distinct values: {values.nunique()}")
    print(values.value_counts().head(5))

Proliferation      2
PgR                1
HER2               1
TrippleNegative    1
dtype: int64

Proliferation | distinct values: 3
Proliferation
1    191
2     72
3     53
Name: count, dtype: int64

PgR | distinct values: 2
PgR
0    188
1    128
Name: count, dtype: int64

HER2 | distinct values: 2
HER2
0    224
1     92
Name: count, dtype: int64

TrippleNegative | distinct values: 2
TrippleNegative
0    209
1    107
Name: count, dtype: int64


In [14]:
# List the categorical features to fill using their training-set modes.
# 列出用训练集众数填补的类别特征。
mode_columns = [
    "ChemoGrade", "HistologyType", "LNStatus", "Proliferation",
    "PgR", "HER2", "TrippleNegative",
]

# Calculate and store one fill value per column using only the training set.
# 只用训练集计算每列的填补值，并保存下来供后续预测使用。
fill_values = X_train_clean[mode_columns].mode().iloc[0]

# Apply the same fill values to the training and validation sets.
# 对训练集和验证集应用同一套填补值。
X_train_filled = X_train_filled.fillna(fill_values)
X_val_filled = X_val_filled.fillna(fill_values)

In [15]:
def encode_gene(features):
    # Encode the three Gene categories as separate 0/1 columns.
    # 把 Gene 的三个类别分别编码成独立的 0/1 列。
    gene = features["Gene"]
    return features.drop(columns="Gene").assign(
        Gene_0=gene.eq(0).astype(int),
        Gene_1=gene.eq(1).astype(int),
        Gene_missing=gene.eq(-1).astype(int),
    )

# Apply the same encoding to both datasets so their columns match.
# 对训练集和验证集使用同一套编码，保证两边的列一致。
X_train_encoded = encode_gene(X_train_filled)
X_val_encoded = encode_gene(X_val_filled)

# Check that the encoded training counts match the original category counts.
# 检查编码后的训练集计数是否和原来的类别计数一致。
print(X_train_encoded[["Gene_0", "Gene_1", "Gene_missing"]].sum())

Gene_0          152
Gene_1           94
Gene_missing     70
dtype: int64


下一步跑 pCR 的第一版逻辑回归。这一格用训练集拟合标准化和模型，再用验证集算 balanced accuracy（平衡准确率）。

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Standardise the features, then fit a class-weighted logistic regression model.
# 先标准化特征，再训练带类别权重的逻辑回归。训练时给较少的阳性样本更高的权重。
pcr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000),
)

# Learn the scaling statistics and model parameters from the training set only.
# 只用训练集计算标准化参数和拟合模型。
pcr_model.fit(X_train_encoded, y_train)

# Predict validation labels using the fitted pipeline.
# 用训练好的流水线预测验证集标签，自动应用同一套标准化参数。
y_val_pred = pcr_model.predict(X_val_encoded)

# Report balanced accuracy and the confusion matrix.
# 输出平衡准确率和混淆矩阵。矩阵的行是真实标签，列是预测标签，顺序均为 0、1。
score = balanced_accuracy_score(y_val, y_val_pred)
print(f"Validation balanced accuracy: {score:.3f}")
print(confusion_matrix(y_val, y_val_pred, labels=[0, 1]))

Validation balanced accuracy: 0.639
[[50 12]
 [ 9  8]]


第一版 pCR 模型跑通了，平衡准确率是 0.639。它识别出了 8/17 个阳性（47.1%），漏掉了 9 个。阴性识别对了 50/62 个（80.6%）。平衡准确率就是这两个比例的平均值。

下一步做一个简单的特征对照实验。用同样的逻辑回归、同样的训练和验证划分，只保留作业要求的 ER、`HER2` 和 Gene，看看分数和漏诊数怎么变。

In [17]:
from sklearn.base import clone

# Keep ER, HER2, and all three encoded columns for Gene.
# 保留 ER、HER2，以及 Gene 独热编码后的三列。
required_features = ["ER", "HER2", "Gene_0", "Gene_1", "Gene_missing"]

# Reuse the same model settings and fit a fresh pipeline on these features.
# 沿用原模型的设置，只用这组特征重新训练。
required_model = clone(pcr_model)
required_model.fit(X_train_encoded[required_features], y_train)

# Evaluate on the same validation patients for a direct comparison.
# 在同一批验证集患者上评分，方便和全特征模型直接比较。
required_pred = required_model.predict(X_val_encoded[required_features])
required_score = balanced_accuracy_score(y_val, required_pred)

print(f"All features: {score:.3f}")
print(f"Required features only: {required_score:.3f}")
print(confusion_matrix(y_val, required_pred, labels=[0, 1]))

All features: 0.639
Required features only: 0.646
[[40 22]
 [ 6 11]]


必留特征模型在这次验证里拿到 0.646，全特征模型是 0.639。它多找出了 3 个阳性（11 对 8），也多把 10 个阴性误判为阳性（22 对 12）。分数只差 0.007，而且验证集只有 17 个阳性。下一步用 5 折分层交叉验证，看看这点差异在不同划分里是否稳定。

新建一格。这里从之前的 X 和 y 开始，把填补、编码和标准化都放进流水线。这样每一折都只从自己的训练部分计算填补值和标准化参数。

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Convert the spreadsheet's missing-value marker to NaN.
# 把表格里的缺失标记 999 转成 NaN，方便后续填补。
X_cv = X.replace([999, "999"], float("nan"))

# Use the same five stratified splits for both feature sets.
# 两组特征共用同样的五折分层划分，方便直接比较。
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Compare all features with the three required features.
# 对比全部特征和三个必留特征。
feature_sets = {
    "All features": X_cv,
    "Required features only": X_cv[["ER", "HER2", "Gene"]],
}

# Store the pipeline for each feature set so it can be refitted later.
# 按特征组保存流水线配置，方便后面用全部有效标注重新训练。
cv_models = {}

for name, features in feature_sets.items():
    # Pick the categorical columns present in this feature set.
    # 找出当前特征组中需要用众数填补的类别列。
    categorical = [
        column for column in ["ER"] + mode_columns
        if column in features.columns
    ]

    # Keep missing Gene values as a separate category and one-hot encode them.
    # 把 Gene 的缺失值记为独立类别，再对三个类别做独热编码。
    gene_pipeline = make_pipeline(
        SimpleImputer(strategy="constant", fill_value=-1),
        OneHotEncoder(categories=[[-1, 0, 1]], handle_unknown="ignore"),
    )

    # Use the mode for categorical columns and the median for other numeric columns.
    # 类别列用众数填补，其余数值列用中位数填补。
    preprocessing = ColumnTransformer(
        [
            ("gene", gene_pipeline, ["Gene"]),
            ("categorical", SimpleImputer(strategy="most_frequent"), categorical),
        ],
        remainder=SimpleImputer(strategy="median"),
        sparse_threshold=0,
    )

    # Combine preprocessing, scaling, and logistic regression into one pipeline.
    # 把预处理、标准化和逻辑回归连成一条流水线。
    model = make_pipeline(
        preprocessing,
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=1000),
    )
    cv_models[name] = model

    # Fit the pipeline on each training fold and score its held-out fold.
    # 每轮只用训练折拟合整条流水线，再用对应的验证折评分。
    scores = cross_val_score(
        model, features, y, cv=cv, scoring="balanced_accuracy"
    )

    # Report the five scores, their mean, and their standard deviation.
    # 输出五折各自的分数、均值和标准差。
    print(f"{name}: {scores.round(3)}")
    print(f"Mean: {scores.mean():.3f}, fold SD: {scores.std():.3f}\n")

All features: [0.732 0.799 0.66  0.681 0.638]
Mean: 0.702, fold SD: 0.058

Required features only: [0.673 0.847 0.644 0.82  0.787]
Mean: 0.754, fold SD: 0.081



只用 ER、`HER2` 和 Gene 的模型，可以先作为 pCR 的候选模型。 它的五折平均平衡准确率是 0.754，比全特征模型的 0.702 高 0.052。它在 5 折里赢了 3 折，分数波动也更大。把两组结果都记下来，后面写报告时可以解释这个取舍。

下一步用这版候选模型跑通 3 条测试样例，并导出两列 CSV。

In [21]:
import pandas as pd
from sklearn.base import clone

# Refit the candidate pipeline on all 395 patients with known pCR labels.
# 用全部 395 条有有效 pCR 标签的记录重新训练候选模型。
pcr_columns = ["ER", "HER2", "Gene"]
pcr_candidate = clone(cv_models["Required features only"])
pcr_candidate.fit(X_cv[pcr_columns], y)

# Load the example file and convert the missing-value marker to NaN.
# 读取测试样例，把特征里的缺失标记 999 转成 NaN。
sample = pd.read_excel("TestDatasetExample.xls")
sample_features = sample[pcr_columns].replace([999, "999"], float("nan"))

# Export one row per patient, with the ID first and the predicted pCR label second.
# 每名患者输出一行，第一列是患者 ID，第二列是预测的 pCR 标签。
sample_predictions = pd.DataFrame({
    "ID": sample["ID"],
    "pCR": pcr_candidate.predict(sample_features),
})
sample_predictions.to_csv("PCRPrediction_example.csv", index=False)
display(sample_predictions)

,ID,pCR
0,TRG002728,0
1,TRG002649,1
2,TRG002628,1
